# Lesson 5.4 — Unit 5 Recap: Execute -> Track

The full loop: move (Execute), judge (Track verdict), monitor (telemetry), read. A healthy pick and a degraded pick, each judged and read.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, plan_reference, execute_reference, track,
    system_monitor, fk_xy, P2, T2)
def pick_layer(seed_world=1, seed_perc=7):
    w = GreenhouseWorld.demo_row(n=6, seed=seed_world)
    dets = model_perception(w, rng=np.random.default_rng(seed_perc))
    _, tgt = understand(dets, w)
    layer = plan_reference(w.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))
    return w, tgt, layer
checks = []
w, tgt, layer = pick_layer()
# healthy
rec_h = execute_reference(layer, telemetry=True)
vh, mh = track(rec_h, target=tgt), system_monitor(rec_h)
print('HEALTHY  -> success=%s reason=%s | peak_err=%.4f peak_effort=%.2f minW=%.4f'
      % (vh['success'], vh['reason'], mh['peak_error'], mh['peak_effort'], mh['min_manipulability']))
# reading: clean pick, nothing to investigate
checks.append(vh['success'] and mh['peak_error'] < 0.01)


HEALTHY  -> success=True reason=ok | peak_err=0.0001 peak_effort=4.61 minW=0.0861


In [2]:
# degraded
def big(t,j): return 60.0 if (j==0 and t>0.3) else 0.0
rec_d = execute_reference(layer, disturbance=big, telemetry=True)
vd, md = track(rec_d, target=tgt), system_monitor(rec_d)
print('DEGRADED -> success=%s reason=%s | peak_err=%.3f peak_effort=%.2f minW=%.4f'
      % (vd['success'], vd['reason'], md['peak_error'], md['peak_effort'], md['min_manipulability']))
# reading: failed on %s; external strain on execution (effort/error up, geometry fine)
checks.append((not vd['success']) and md['peak_effort'] > mh['peak_effort'])
# health != success holds: degraded verdict tracks the elevated telemetry
checks.append(vd['reason'] in ('final_error','rms','pose'))
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


DEGRADED -> success=False reason=final_error | peak_err=2.203 peak_effort=70.02 minW=0.0861
3/3 checks passed.
All checks passed.
